# This Notebook estimates the model

## Settings

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
from scipy.optimize import minimize

import DynamicTimeAllocationModel

# c++ settings
do_compile = True
threads = 64

import os
os.environ.pop('NoDefaultCurrentDirectoryInExePath', None)

do_reinstall_nlopt = False # If problems with NLOPT during re-compilation of c++ files, then try re-installing NLOPT by swithcing this to True and delete the folder "nlopt-2.4.2-dll64" in the cppfuncs folder before running this notebook.
if do_reinstall_nlopt:
    from EconModel import cpptools
    cpptools.setup_nlopt(folder='cppfuncs/', do_print=False,download=False,unzip=True)

In [2]:
# setup model
settings = { 
       # technical settings
       'threads':threads,
       'do_multistart': False,
       'do_egm': True,
       'interp_method': 'linear',
       'interp_inverse': True,
       'precompute_intratemporal': True,
       'centered_gradient': True,
       'bargaining': 'limited',
}


model = DynamicTimeAllocationModel.HouseholdModelClass(par=settings) 
model.link_to_cpp(force_compile=do_compile)

## Empirical Moments to Match

In [3]:
# all moments listed here will be used in estimation. Comment out those you do not want to use.
datamoms = dict()

# wages
datamoms['wage_level_w_25_34'] = 40.1
datamoms['wage_level_w_35_44'] = 49.3
datamoms['wage_level_m_25_34'] = 50.3
datamoms['wage_level_m_35_44'] = 67.8

# employment rates
datamoms['employment_rate_w_35_44'] = 64.0
datamoms['employment_rate_m_35_44'] = 88.0
datamoms['work_hours_w'] = 4.41*365 / 52.0  # daily hours to weekly hours
datamoms['work_hours_m'] = 5.7*365 / 52.0  # daily hours to weekly hours

# # consumption
datamoms['consumption'] = 42.716
datamoms['consumption_90_10_ratio'] = 3.33 * 1.0954

# # marriage and divorce rates
datamoms['marriage_rate_35_44'] = 69.0

# Mazzocco moments
datamoms['home_prod_w'] = (2.23+0.75+1.47+0.08) * 365/ 52
datamoms['home_prod_m'] = (1.6+0.54+0.88+0.1) * 365/ 52


# weights
weights = dict()
for mom in ('consumption_90_10_ratio',):
    weights[mom] = 10.0
    

## Parameters to estimate

In [4]:
# parameters to estimate
estpars = {
    # Wages
    'mu': {'guess':2.3678,'lower':0.1,'upper':3.00}, 
    'mu_mult': {'guess':1.1126,'lower':1.0,'upper':3.0},
    'gamma': {'guess':0.1237,'lower':0.001,'upper':0.50},
    'gamma_mult': {'guess':1.7611,'lower':1.0,'upper':3.0},
    'sigma_mu': {'guess':0.5613,'lower':0.001,'upper':1.0},
    
    # Disutility from work
    'eta': {'guess':0.9033,'lower':0.1,'upper':5.0},
    'eta_mult': {'guess':0.8877,'lower':0.3,'upper':3.0},
    'phi': {'guess':4.4732,'lower':0.1,'upper':5.0},
    'phi_mult': {'guess':1.0855,'lower':0.3,'upper':3.0},
    
    # Home production
    'alpha': {'guess':0.9608,'lower':0.1,'upper':1.9},
    'pi': {'guess':0.6144,'lower':0.1,'upper':0.9},
    'lambda_': {'guess':5.7527,'lower':0.1,'upper':30.0},
    
    # # Match quality
    'sigma_love': {'guess':3.7895,'lower':0.01,'upper':20.5},
}

## setup initial guess 

In [5]:
# check bounds
bounds_ok = True
for key in estpars.keys():
    if estpars[key]['guess']<estpars[key]['lower']:
        print(key,' lower',estpars[key]['guess'])
        bounds_ok = False
    
    if estpars[key]['guess']>estpars[key]['upper']:
        print(key,' upper',estpars[key]['guess'])
        bounds_ok = False

if not bounds_ok:
    stop

In [6]:
# check initial guess
theta_init = np.array([estpars[key]['guess'] for key in estpars.keys()])
obj_init = model.obj_func(theta_init, estpars, datamoms, weights, do_print=True)

Parameters:
  mu             : 2.3678 (init: 2.3678)
  mu_mult        : 1.1126 (init: 1.1126)
  gamma          : 0.1237 (init: 0.1237)
  gamma_mult     : 1.7611 (init: 1.7611)
  sigma_mu       : 0.5613 (init: 0.5613)
  eta            : 0.9033 (init: 0.9033)
  eta_mult       : 0.8877 (init: 0.8877)
  phi            : 4.4732 (init: 4.4732)
  phi_mult       : 1.0855 (init: 1.0855)
  alpha          : 0.9608 (init: 0.9608)
  pi             : 0.6144 (init: 0.6144)
  lambda_        : 5.7527 (init: 5.7527)
  sigma_love     : 3.7895 (init: 3.7895)
Moments:
  wage_level_w_25_34       : sim: 37.6246, data: 40.1000
  wage_level_w_35_44       : sim: 51.8620, data: 49.3000
  wage_level_m_25_34       : sim: 50.0506, data: 50.3000
  wage_level_m_35_44       : sim: 67.0226, data: 67.8000
  employment_rate_w_35_44  : sim: 63.8269, data: 64.0000
  employment_rate_m_35_44  : sim: 88.4375, data: 88.0000
  work_hours_w             : sim: 27.9627, data: 30.9548
  work_hours_m             : sim: 36.5938, data

## Estimate model

In [7]:
# Estimate model using nelder-mead algorithm
do_print = True
res = minimize(model.obj_func, theta_init, args=(estpars, datamoms,weights,do_print), method='Nelder-Mead',
               options={'xatol': 1e-3, 'fatol': 1e-3, 'disp': True, 'maxiter':500, 'maxfev':500})


Parameters:
  mu             : 2.3678 (init: 2.3678)
  mu_mult        : 1.1126 (init: 1.1126)
  gamma          : 0.1237 (init: 0.1237)
  gamma_mult     : 1.7611 (init: 1.7611)
  sigma_mu       : 0.5613 (init: 0.5613)
  eta            : 0.9033 (init: 0.9033)
  eta_mult       : 0.8877 (init: 0.8877)
  phi            : 4.4732 (init: 4.4732)
  phi_mult       : 1.0855 (init: 1.0855)
  alpha          : 0.9608 (init: 0.9608)
  pi             : 0.6144 (init: 0.6144)
  lambda_        : 5.7527 (init: 5.7527)
  sigma_love     : 3.7895 (init: 3.7895)
Moments:
  wage_level_w_25_34       : sim: 37.6246, data: 40.1000
  wage_level_w_35_44       : sim: 51.8620, data: 49.3000
  wage_level_m_25_34       : sim: 50.0506, data: 50.3000
  wage_level_m_35_44       : sim: 67.0226, data: 67.8000
  employment_rate_w_35_44  : sim: 63.8269, data: 64.0000
  employment_rate_m_35_44  : sim: 88.4375, data: 88.0000
  work_hours_w             : sim: 27.9627, data: 30.9548
  work_hours_m             : sim: 36.5938, data

Parameters:
  mu             : 2.4862 (init: 2.3678)
  mu_mult        : 1.1126 (init: 1.1126)
  gamma          : 0.1237 (init: 0.1237)
  gamma_mult     : 1.7611 (init: 1.7611)
  sigma_mu       : 0.5613 (init: 0.5613)
  eta            : 0.9033 (init: 0.9033)
  eta_mult       : 0.8877 (init: 0.8877)
  phi            : 4.4732 (init: 4.4732)
  phi_mult       : 1.0855 (init: 1.0855)
  alpha          : 0.9608 (init: 0.9608)
  pi             : 0.6144 (init: 0.6144)
  lambda_        : 5.7527 (init: 5.7527)
  sigma_love     : 3.7895 (init: 3.7895)
Moments:
  wage_level_w_25_34       : sim: 43.6811, data: 40.1000
  wage_level_w_35_44       : sim: 58.6674, data: 49.3000
  wage_level_m_25_34       : sim: 53.4892, data: 50.3000
  wage_level_m_35_44       : sim: 74.3338, data: 67.8000
  employment_rate_w_35_44  : sim: 62.1446, data: 64.0000
  employment_rate_m_35_44  : sim: 93.4340, data: 88.0000
  work_hours_w             : sim: 27.7722, data: 30.9548
  work_hours_m             : sim: 38.1327, data

Parameters:
  mu             : 2.3678 (init: 2.3678)
  mu_mult        : 1.1682 (init: 1.1126)
  gamma          : 0.1237 (init: 0.1237)
  gamma_mult     : 1.7611 (init: 1.7611)
  sigma_mu       : 0.5613 (init: 0.5613)
  eta            : 0.9033 (init: 0.9033)
  eta_mult       : 0.8877 (init: 0.8877)
  phi            : 4.4732 (init: 4.4732)
  phi_mult       : 1.0855 (init: 1.0855)
  alpha          : 0.9608 (init: 0.9608)
  pi             : 0.6144 (init: 0.6144)
  lambda_        : 5.7527 (init: 5.7527)
  sigma_love     : 3.7895 (init: 3.7895)
Moments:
  wage_level_w_25_34       : sim: 43.3469, data: 40.1000
  wage_level_w_35_44       : sim: 53.6991, data: 49.3000
  wage_level_m_25_34       : sim: 53.2299, data: 50.3000
  wage_level_m_35_44       : sim: 73.4069, data: 67.8000
  employment_rate_w_35_44  : sim: 52.2344, data: 64.0000
  employment_rate_m_35_44  : sim: 95.0166, data: 88.0000
  work_hours_w             : sim: 25.1205, data: 30.9548
  work_hours_m             : sim: 38.5869, data

Parameters:
  mu             : 2.3678 (init: 2.3678)
  mu_mult        : 1.1126 (init: 1.1126)
  gamma          : 0.1299 (init: 0.1237)
  gamma_mult     : 1.7611 (init: 1.7611)
  sigma_mu       : 0.5613 (init: 0.5613)
  eta            : 0.9033 (init: 0.9033)
  eta_mult       : 0.8877 (init: 0.8877)
  phi            : 4.4732 (init: 4.4732)
  phi_mult       : 1.0855 (init: 1.0855)
  alpha          : 0.9608 (init: 0.9608)
  pi             : 0.6144 (init: 0.6144)
  lambda_        : 5.7527 (init: 5.7527)
  sigma_love     : 3.7895 (init: 3.7895)
Moments:
  wage_level_w_25_34       : sim: 37.9453, data: 40.1000
  wage_level_w_35_44       : sim: 52.7332, data: 49.3000
  wage_level_m_25_34       : sim: 49.2425, data: 50.3000
  wage_level_m_35_44       : sim: 67.6253, data: 67.8000
  employment_rate_w_35_44  : sim: 62.8817, data: 64.0000
  employment_rate_m_35_44  : sim: 91.8477, data: 88.0000
  work_hours_w             : sim: 27.8527, data: 30.9548
  work_hours_m             : sim: 37.6525, data

Parameters:
  mu             : 2.3678 (init: 2.3678)
  mu_mult        : 1.1126 (init: 1.1126)
  gamma          : 0.1237 (init: 0.1237)
  gamma_mult     : 1.8492 (init: 1.7611)
  sigma_mu       : 0.5613 (init: 0.5613)
  eta            : 0.9033 (init: 0.9033)
  eta_mult       : 0.8877 (init: 0.8877)
  phi            : 4.4732 (init: 4.4732)
  phi_mult       : 1.0855 (init: 1.0855)
  alpha          : 0.9608 (init: 0.9608)
  pi             : 0.6144 (init: 0.6144)
  lambda_        : 5.7527 (init: 5.7527)
  sigma_love     : 3.7895 (init: 3.7895)
Moments:
  wage_level_w_25_34       : sim: 38.3878, data: 40.1000
  wage_level_w_35_44       : sim: 52.3175, data: 49.3000
  wage_level_m_25_34       : sim: 48.7553, data: 50.3000
  wage_level_m_35_44       : sim: 67.3053, data: 67.8000
  employment_rate_w_35_44  : sim: 61.5599, data: 64.0000
  employment_rate_m_35_44  : sim: 92.7058, data: 88.0000
  work_hours_w             : sim: 27.4144, data: 30.9548
  work_hours_m             : sim: 37.9182, data

...

Parameters:
  mu             : 2.3782 (init: 2.3678)
  mu_mult        : 1.1285 (init: 1.1126)
  gamma          : 0.1091 (init: 0.1237)
  gamma_mult     : 1.8162 (init: 1.7611)
  sigma_mu       : 0.5559 (init: 0.5613)
  eta            : 0.9499 (init: 0.9033)
  eta_mult       : 0.8966 (init: 0.8877)
  phi            : 4.3952 (init: 4.4732)
  phi_mult       : 1.0968 (init: 1.0855)
  alpha          : 0.9271 (init: 0.9608)
  pi             : 0.6296 (init: 0.6144)
  lambda_        : 5.9770 (init: 5.7527)
  sigma_love     : 3.8537 (init: 3.7895)
Moments:
  wage_level_w_25_34       : sim: 38.4634, data: 40.1000
  wage_level_w_35_44       : sim: 50.5640, data: 49.3000
  wage_level_m_25_34       : sim: 51.4284, data: 50.3000
  wage_level_m_35_44       : sim: 66.7933, data: 67.8000
  employment_rate_w_35_44  : sim: 63.6310, data: 64.0000
  employment_rate_m_35_44  : sim: 88.8434, data: 88.0000
  work_hours_w             : sim: 27.7468, data: 30.9548
  work_hours_m             : sim: 36.5089, data

Parameters:
  mu             : 2.3782 (init: 2.3678)
  mu_mult        : 1.1285 (init: 1.1126)
  gamma          : 0.1091 (init: 0.1237)
  gamma_mult     : 1.8162 (init: 1.7611)
  sigma_mu       : 0.5559 (init: 0.5613)
  eta            : 0.9499 (init: 0.9033)
  eta_mult       : 0.8966 (init: 0.8877)
  phi            : 4.3952 (init: 4.4732)
  phi_mult       : 1.0968 (init: 1.0855)
  alpha          : 0.9271 (init: 0.9608)
  pi             : 0.6296 (init: 0.6144)
  lambda_        : 5.9769 (init: 5.7527)
  sigma_love     : 3.8535 (init: 3.7895)
Moments:
  wage_level_w_25_34       : sim: 38.4642, data: 40.1000
  wage_level_w_35_44       : sim: 50.5616, data: 49.3000
  wage_level_m_25_34       : sim: 51.4291, data: 50.3000
  wage_level_m_35_44       : sim: 66.7906, data: 67.8000
  employment_rate_w_35_44  : sim: 63.6324, data: 64.0000
  employment_rate_m_35_44  : sim: 88.8418, data: 88.0000
  work_hours_w             : sim: 27.7467, data: 30.9548
  work_hours_m             : sim: 36.5082, data

Parameters:
  mu             : 2.3782 (init: 2.3678)
  mu_mult        : 1.1286 (init: 1.1126)
  gamma          : 0.1091 (init: 0.1237)
  gamma_mult     : 1.8164 (init: 1.7611)
  sigma_mu       : 0.5559 (init: 0.5613)
  eta            : 0.9499 (init: 0.9033)
  eta_mult       : 0.8967 (init: 0.8877)
  phi            : 4.3955 (init: 4.4732)
  phi_mult       : 1.0968 (init: 1.0855)
  alpha          : 0.9271 (init: 0.9608)
  pi             : 0.6296 (init: 0.6144)
  lambda_        : 5.9769 (init: 5.7527)
  sigma_love     : 3.8534 (init: 3.7895)
Moments:
  wage_level_w_25_34       : sim: 38.4683, data: 40.1000
  wage_level_w_35_44       : sim: 50.5577, data: 49.3000
  wage_level_m_25_34       : sim: 51.4308, data: 50.3000
  wage_level_m_35_44       : sim: 66.7885, data: 67.8000
  employment_rate_w_35_44  : sim: 63.6316, data: 64.0000
  employment_rate_m_35_44  : sim: 88.8425, data: 88.0000
  work_hours_w             : sim: 27.7455, data: 30.9548
  work_hours_m             : sim: 36.5076, data

Parameters:
  mu             : 2.3782 (init: 2.3678)
  mu_mult        : 1.1285 (init: 1.1126)
  gamma          : 0.1091 (init: 0.1237)
  gamma_mult     : 1.8163 (init: 1.7611)
  sigma_mu       : 0.5559 (init: 0.5613)
  eta            : 0.9499 (init: 0.9033)
  eta_mult       : 0.8966 (init: 0.8877)
  phi            : 4.3953 (init: 4.4732)
  phi_mult       : 1.0968 (init: 1.0855)
  alpha          : 0.9271 (init: 0.9608)
  pi             : 0.6296 (init: 0.6144)
  lambda_        : 5.9768 (init: 5.7527)
  sigma_love     : 3.8535 (init: 3.7895)
Moments:
  wage_level_w_25_34       : sim: 38.4658, data: 40.1000
  wage_level_w_35_44       : sim: 50.5600, data: 49.3000
  wage_level_m_25_34       : sim: 51.4288, data: 50.3000
  wage_level_m_35_44       : sim: 66.7891, data: 67.8000
  employment_rate_w_35_44  : sim: 63.6322, data: 64.0000
  employment_rate_m_35_44  : sim: 88.8413, data: 88.0000
  work_hours_w             : sim: 27.7479, data: 30.9548
  work_hours_m             : sim: 36.5079, data

C:\Users\zbk883\AppData\Local\Temp\4\ipykernel_25840\1613049469.py:3: RuntimeWarning: Maximum number of function evaluations has been exceeded.
  res = minimize(model.obj_func, theta_init, args=(estpars, datamoms,weights,do_print), method='Nelder-Mead',


In [ ]:
# final solution: objective, parameters and simulated vs. data moments at the optimum
obj_final = model.obj_func(res.x, estpars, datamoms, weights, do_print=True)

In [8]:
model.save_par('calibrated_par')